## 获取任务sql

In [6]:
from voydstools.common.DataServiceAPI import DataServiceHttp

In [7]:
# For test
x_app_key = '50214295'
secret_key = 'e467683796fe3ae3792bf83fa0ef2796bb47dd90'
api_name = "get_tabel_sql_temp_sql_df"

api = DataServiceHttp(x_app_key, secret_key)
api.set_api(api_name)

query_data = {
  "dt": "2024-09-13",
  "project_id": "3100",
  "task_id": "31041901"
}

response = api.post(query_data)
print(response)

def get_sql_from_task_id(project_id, task_id):
    query_data = {
        "dt": "2024-09-13",
        "project_id": project_id,
        "task_id": task_id
    }
    response = api.post(query_data)
    return response[0]['file_content']


请求：http://10.88.128.15:8000/dataservice/gateway/v1/api/get_tabel_sql_temp_sql_df，数据：{'dt': '2024-09-13', 'project_id': '3100', 'task_id': '31041901', 'pageSize': 5000, 'page': 1}
[{'file_version': 1, 'file_content': "--@exclude_dependency=sparklingwater.gateway_daily_hudi_ods_brp_--SPARK_SQL_brp_--********************************************************************--_brp_--author:zhengzong_brp_--create time:2024-08-01 16:26:40_brp_--desc:雨刷器档位_brp_--remind:请在资源引用中添加需要引用的资源_brp_--********************************************************************--_brp_--- DROP TABLE IF EXISTS app_trip_rt_wipe;_brp_-- create table if not exists app_trip_rt_wiper_brp_-- (_brp_--     rigel_meta_trip_id string_brp_--     ,wiper_speed double_brp_--     ,wiper_speed_cn string_brp_-- )_brp_-- partitioned by_brp_-- (_brp_--     rigel_meta_event_date string comment '分区'_brp_-- )_brp_-- ;_brp__brp__brp_-- insert overwrite table app_trip_rt_wiper partition (rigel_meta_event_date = '${BIZ_DATE_LINE}')_brp_-- sele

In [3]:
sql = get_sql_from_task_id('3100', '31041901')
sql

请求：http://10.88.128.15:8000/dataservice/gateway/v1/api/get_tabel_sql_temp_sql_df，数据：{'dt': '2024-09-13', 'project_id': '3100', 'task_id': '31041901', 'pageSize': 5000, 'page': 1}


"--@exclude_dependency=sparklingwater.gateway_daily_hudi_ods_brp_--SPARK_SQL_brp_--********************************************************************--_brp_--author:zhengzong_brp_--create time:2024-08-01 16:26:40_brp_--desc:雨刷器档位_brp_--remind:请在资源引用中添加需要引用的资源_brp_--********************************************************************--_brp_--- DROP TABLE IF EXISTS app_trip_rt_wipe;_brp_-- create table if not exists app_trip_rt_wiper_brp_-- (_brp_--     rigel_meta_trip_id string_brp_--     ,wiper_speed double_brp_--     ,wiper_speed_cn string_brp_-- )_brp_-- partitioned by_brp_-- (_brp_--     rigel_meta_event_date string comment '分区'_brp_-- )_brp_-- ;_brp__brp__brp_-- insert overwrite table app_trip_rt_wiper partition (rigel_meta_event_date = '${BIZ_DATE_LINE}')_brp_-- select_brp_--     rigel_meta_trip_id,_brp_--     max(vehicle_detail_info__wiper_speed) as wiper_speed,_brp_--     case   _brp_--         when sum(case when vehicle_detail_info__wiper_speed in (5,6) then 1 else 0 end) > 3

## 字段血缘关系

In [4]:
from sqllineage.runner import LineageRunner
# SUPPORTED_DIALECTS = list(dialect.label for dialect in dialect_readout())
# SUPPORTED_DIALECTS

In [4]:
sql = """
CREATE TABLE if not exists  `dim_rt_trip_odd_info_df`(
  `trip_id` string COMMENT 'trip_id', 
  `trip_odd` string COMMENT 'trip_odd',
  `create_date` string COMMENT 'create_date'
  )
PARTITIONED BY ( 
  `dt` string)
;

WITH app_rt_trip_issue_detail_hf AS (
    SELECT DISTINCT
        trip_id,
        region,
        create_date
    FROM 
        vgds.app_rt_trip_issue_detail_hf
    WHERE 
        dt = '2024-09-11-15'
),

trip_odd_data AS (
    SELECT DISTINCT
        b.trip_id, 
        CASE
            WHEN b.create_date <= a.create_static_date THEN a.trip_odd 
            ELSE 
                CASE
                    WHEN b.region like '%guangzhou%' or b.region like '%广州%' THEN 'ODD2'
                    WHEN b.region like '%beijing_yizhuang%' or b.region like '%北京亦庄%' THEN 'ODD2'
                    WHEN b.region like '%shanghai%' or b.region like '%上海%' THEN 'ODD3'
                    ELSE 'ODD-OTHER'
                END
        END AS trip_odd,
        b.create_date
    FROM 
        vgds.dim_rt_trip_odd_static_df a
    FULL OUTER JOIN 
        app_rt_trip_issue_detail_hf b
    ON a.trip_id = b.trip_id
)

INSERT OVERWRITE TABLE dim_rt_trip_odd_info_df partition (dt = '2024-09-11-15')
select 
    trip_id
    ,trip_odd 
    ,create_date
from
    trip_odd_data;
"""


In [3]:
from sqllineage.runner import LineageRunner
result = LineageRunner(sql,dialect='non-validating')
print(result)

/tmp/ipykernel_2598974/2817020059.py:2: DeprecationWarning: dialect `non-validating` is deprecated, use `ansi` or dialect of your SQL instead. `non-validating` will be completely removed in v1.6.x
  result = LineageRunner(sql,dialect='non-validating')


Statements(#): 2
Source Tables:
    vgds.dim_rt_issue_topic_view_hf
    vgds.dim_rt_trip_distance_accumulated_df
    vgds.dim_rt_version_date_range_df
    vgds.dwd_rt3_task_order_package_case_order_hf
    vgds.dwd_rt_issue_with_merged_topic_detail_hf
    vgds.dwd_rt_trip_info_hf
    vgds.dwd_ssevent_data_quality_issue_detail_hf
    vgds.ods_rt_issue_info_1_hf
Target Tables:
    <default>.app_rt_trip_issue_detail_hf



In [5]:
result.draw()

 * SQLLineage Running on http://localhost:5001/?e=%0Acreate+table+if+not+exists+%60app_rt_trip_issue_detail_hf%60%0A%28%0A++++%60car_id%60+string+COMMENT+%27%E8%BD%A6%E8%BE%86%E7%BC%96%E5%8F%B7%27%0A++++%2C%60trip_comment%60+string+COMMENT+%27comment%27%0A++++%2C%60country%60+bigint+COMMENT+%271%3Acn+2%3Aus%27%0A++++%2C%60driver_name%60+string+COMMENT+%27%E9%A9%BE%E9%A9%B6%E5%91%98%E5%90%8D%E7%A7%B0%27%0A++++%2C%60bag_trip_end_timestamp%60+bigint+COMMENT+%27bag_trip_end_timestamp%27%0A++++%2C%60region%60+string+COMMENT+%27%E5%9C%B0%E5%8C%BA%27%0A++++%2C%60bag_trip_start_timestamp%60+bigint+COMMENT+%27bag_trip_start_timestamp%27%0A++++%2C%60trip_id%60+string+COMMENT+%27trip_id%27%0A++++%2C%60update_time%60+string+COMMENT+%27%E6%9B%B4%E6%96%B0%E6%97%B6%E9%97%B4%27%0A++++%2C%60user_name%60+string+COMMENT+%27%E5%AE%89%E5%85%A8%E5%91%98%E5%90%8D%E7%A7%B0%27%0A++++%2C%60test_version%60+string+COMMENT+%27%E8%87%AA%E5%8A%A8%E9%A9%BE%E9%A9%B6%E7%89%88%E6%9C%AC%27%0A++++%2C%60road_test_type%60+b

127.0.0.1 - - [16/Sep/2024 12:21:35] "GET / HTTP/1.1" 200 755
127.0.0.1 - - [16/Sep/2024 12:21:35] "GET / HTTP/1.1" 200 755
127.0.0.1 - - [16/Sep/2024 12:21:35] "GET /static/css/main.84d1d546.css HTTP/1.1" 200 65835
127.0.0.1 - - [16/Sep/2024 12:21:35] "GET /static/js/main.3b88c6e3.js HTTP/1.1" 200 3184857
127.0.0.1 - - [16/Sep/2024 12:21:35] "GET /static/css/main.84d1d546.css HTTP/1.1" 200 65835
127.0.0.1 - - [16/Sep/2024 12:21:35] "GET /static/js/main.3b88c6e3.js HTTP/1.1" 200 3184857
127.0.0.1 - - [16/Sep/2024 12:21:36] "GET /editor.worker.js HTTP/1.1" 200 121291
127.0.0.1 - - [16/Sep/2024 12:21:36] "GET /static/js/333.140e3456.chunk.js HTTP/1.1" 200 18267
127.0.0.1 - - [16/Sep/2024 12:21:36] "POST /lineage HTTP/1.1" 200 124
127.0.0.1 - - [16/Sep/2024 12:21:36] "POST /directory HTTP/1.1" 200 235
127.0.0.1 - - [16/Sep/2024 12:21:36] "POST /lineage HTTP/1.1" 200 124
127.0.0.1 - - [16/Sep/2024 12:21:36] "GET /manifest.json HTTP/1.1" 200 494
127.0.0.1 - - [16/Sep/2024 12:21:36] "GET /fa

routes: {'/lineage': <function lineage at 0x7cd6e39172e0>, '/script': <function script at 0x7cd6e3917380>, '/directory': <function directory at 0x7cd6e3917420>}
data: {'verbose': '==========\nSummary:\nStatements(#): 0\nSource Tables:\n    \nTarget Tables:\n    \n', 'dag': [], 'column': []}
routes: {'/lineage': <function lineage at 0x7cd6e39172e0>, '/script': <function script at 0x7cd6e3917380>, '/directory': <function directory at 0x7cd6e3917420>}
data: {'id': '/home/zhengzong/workspace/DS/HiveTrace/HiveTrace/sqllineage/data', 'name': 'data', 'is_dir': True, 'children': [{'id': '/home/zhengzong/workspace/DS/HiveTrace/HiveTrace/sqllineage/data/tpcds', 'name': 'tpcds', 'is_dir': True}]}
routes: {'/lineage': <function lineage at 0x7cd6e39172e0>, '/script': <function script at 0x7cd6e3917380>, '/directory': <function directory at 0x7cd6e3917420>}
data: {'verbose': '==========\nSummary:\nStatements(#): 0\nSource Tables:\n    \nTarget Tables:\n    \n', 'dag': [], 'column': []}
routes: {'/li

127.0.0.1 - - [16/Sep/2024 12:21:36] "GET /manifest.json HTTP/1.1" 200 494
127.0.0.1 - - [16/Sep/2024 12:21:36] "GET /favicon.ico HTTP/1.1" 200 3870
127.0.0.1 - - [16/Sep/2024 12:21:36] "GET /logo192.png HTTP/1.1" 200 5347
127.0.0.1 - - [16/Sep/2024 12:22:17] "GET /static/css/main.84d1d546.css HTTP/1.1" 200 65835
127.0.0.1 - - [16/Sep/2024 12:22:18] "GET /static/js/main.3b88c6e3.js.map HTTP/1.1" 200 12881299
127.0.0.1 - - [16/Sep/2024 12:22:18] "GET /manifest.json HTTP/1.1" 200 494
127.0.0.1 - - [16/Sep/2024 12:22:18] "GET /static/js/333.140e3456.chunk.js.map HTTP/1.1" 200 45386
127.0.0.1 - - [16/Sep/2024 12:22:18] "GET /editor.worker.js.map HTTP/1.1" 200 576256
127.0.0.1 - - [16/Sep/2024 12:22:18] "GET /static/css/main.84d1d546.css.map HTTP/1.1" 200 145611


KeyboardInterrupt: 

## vgds库sql代码查询

In [6]:
sql = get_sql_from_task_id('3100', '31041901')

请求：http://10.88.128.15:8000/dataservice/gateway/v1/api/get_tabel_sql_temp_sql_df，数据：{'dt': '2024-09-13', 'project_id': '3100', 'task_id': '31041901', 'pageSize': 5000, 'page': 1}


In [29]:
def format_sql(sql_str):
    # 替换多个空格为单个空格，避免冗余
    formatted_sql = ' '.join(sql_str.split())

    # 按照指定的符号、关键词等进行换行和格式化
    formatted_sql = formatted_sql.replace('_brp_ _', '\n,')
    formatted_sql = formatted_sql.replace('_brp_', '\n') \
                                 .replace('_ ', ',') \
     
    # formatted_sql = formatted_sql.replace('--', '\n--')\
                                #  .replace('create table if not exists', '\ncreate table if not exists')\
                                #  .replace('partitioned by', '\npartitioned by')\
                                #  .replace('select', '\nselect')\
                                #  .replace('from', '\nfrom')\
                                #  .replace('where', '\nwhere')\
                                #  .replace('group by', '\ngroup by')\
                                #  .replace('insert overwrite table', '\ninsert overwrite table')\
                                #  .replace('with', '\nwith')\
                                #  .replace('case', '\n    case')\
                                #  .replace('end as', '\n    end as')\
                                #  .replace('when', '\n        when')\
                                #  .replace('else', '\n        else')\

    # 去掉可能多余的空格
    formatted_sql = formatted_sql.replace(' ;', ';')
    # 如果最后不是分号结尾，加上分号
    if formatted_sql[-1] != ';':
        formatted_sql += ';'

    return formatted_sql

In [8]:
formatted_sql = format_sql(sql)

result = LineageRunner(formatted_sql,dialect='non-validating')
print(result)

Statements(#): 2
Source Tables:
    sparklingwater.gateway_daily_hudi_ods
Target Tables:
    <default>.app_trip_rt_wiper_df_all



/tmp/ipykernel_1596794/887143189.py:3: DeprecationWarning: dialect `non-validating` is deprecated, use `ansi` or dialect of your SQL instead. `non-validating` will be completely removed in v1.6.x
  result = LineageRunner(formatted_sql,dialect='non-validating')


In [11]:
result.draw()

 * SQLLineage Running on http://localhost:5001/?e=--%40exclude_dependency%3Dsparklingwater.gateway_daily_hudi_ods%0A--SPARK_SQL%0A--%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A--%0A--author%3Azhengzong%0A--create+time%3A2024-08-01+16%3A26%3A40%0A--desc%3A%E9%9B%A8%E5%88%B7%E5%99%A8%E6%A1%A3%E4%BD%8D%0A--remind%3A%E8%AF%B7%E5%9C%A8%E8%B5%84%E6%BA%90%E5%BC%95%E7%94%A8%E4%B8%AD%E6%B7%BB%E5%8A%A0%E9%9C%80%E8%A6%81%E5%BC%95%E7%94%A8%E7%9A%84%E8%B5%84%E6%BA%90%0A--%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A--%0A---+DROP+TABLE+IF+EXISTS+app_trip_rt_wipe%3B%0A--+create+table+if+not+exists+app_trip_rt_wiper%0A--+%28%0A--+rigel_meta_trip_id+string%0A--+%2Cwiper_speed+double%0A--+%

127.0.0.1 - - [14/Sep/2024 21:37:16] "GET /?e=--%40exclude_dependency%3Dsparklingwater.gateway_daily_hudi_ods%0A--SPARK_SQL%0A--%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A--%0A--author%3Azhengzong%0A--create+time%3A2024-08-01+16%3A26%3A40%0A--desc%3A%E9%9B%A8%E5%88%B7%E5%99%A8%E6%A1%A3%E4%BD%8D%0A--remind%3A%E8%AF%B7%E5%9C%A8%E8%B5%84%E6%BA%90%E5%BC%95%E7%94%A8%E4%B8%AD%E6%B7%BB%E5%8A%A0%E9%9C%80%E8%A6%81%E5%BC%95%E7%94%A8%E7%9A%84%E8%B5%84%E6%BA%90%0A--%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A--%0A---+DROP+TABLE+IF+EXISTS+app_trip_rt_wipe%3B%0A--+create+table+if+not+exists+app_trip_rt_wiper%0A--+%28%0A--+rigel_meta_trip_id+string%0A--+%2Cwiper_speed+double%0A--+%2Cwi

KeyboardInterrupt: 

### 本地表导入

In [40]:
import pandas as pd
import numpy as np

sql_code_df = pd.read_csv('/home/zhengzong/workspace/DS/sql任务表查询.csv',engine='python',encoding='gbk')

## 根据file_name筛选
def select_sql_from_file_name(df):
    # 过滤.csv / .xlsx / .xls .zip结尾的
    df = df[~df['file_name'].str.contains('.csv|.xlsx|.xls|.zip')]
    # 过滤Hive2ClickHous、Cooper2Hive、Hive2MySQL、
    df = df[~df['file_name'].str.contains('Hive2ClickHous|Cooper2Hive|Hive2MySQL|MySQL2Hive|MysqlToHive|hive2ck')]
    # 过滤掉test的
    df = df[~df['file_name'].str.contains('test|tmp')]
    # 过滤掉query的
    df = df[~df['file_name'].str.contains('query')]
    # 过滤掉空的file_content
    df = df[df['file_content'].notnull()]
    # 过滤掉.txt/.conf结尾的
    df = df[~df['file_name'].str.contains('.txt|.conf')]
    # 过滤掉datalinkapi_开头的
    df = df[~df['file_name'].str.contains('datalinkapi_')]
    # 过滤掉alter开头的
    df = df[~df['file_name'].str.contains('alter')]
    return df

sql_code_df = select_sql_from_file_name(sql_code_df)

In [41]:
sql_code_df

,id,file_id,file_version,file_content,commit_time,commit_user,schedule_content,schedule_uuid,file_type,use_type,file_desc,file_name,status,change_type,is_current_prod,project_id,file_property_content,parent_file_id
3,35846765,18136735,2,--SPARK_SQL_brp_--****************************...,2022-07-26 19:09:22.000,sherlockshen,"{""alarmMode"":""""_""alarmOdinGroup"":""""_""alarmRece...",vgds.dwd_autolabel_tp_lane_change_july,104,2,dwd_autolabel_tp_lane_change_july,dwd_autolabel_tp_lane_change_july,101,1,1,3100,"{""commitStatus"":0_""countryGroupCode"":""""_""count...",0
5,35864359,18133447,1,from pyspark.sql.functions import udf_brp_from...,2022-07-26 19:34:50.000,zhonghao_i,NaN,NaN,212,4,NaN,get_latlon_by_pose.py,101,0,1,3100,"{""commitStatus"":0_""countryGroupCode"":""""_""count...",0
7,40977787,19959513,6,#!/usr/bin/env python_brp_# -*- coding: utf-8 ...,2022-12-27 17:12:38.000,yanranhan,"{""alarmDchatGroup"":""""_""alarmMode"":""""_""alarmOdi...",vgds.route_grade_odd,105,2,for us,route_grade_odd,101,1,1,3100,"{""commitStatus"":0_""countryGroupCode"":""""_""count...",0
8,41027893,20102029,3,#!/usr/bin/env python_brp_# -*- coding: utf-8 ...,2022-12-30 10:43:56.000,jimmylimao,"{""alarmDchatGroup"":""""_""alarmMode"":""""_""alarmOdi...",vgds.ego_lean_one_side_2,105,2,2,ego_lean_one_side_2,101,1,1,3100,"{""commitStatus"":0_""countryGroupCode"":""""_""count...",0
9,41037721,14243975,20,--SPARK_SQL_brp_--****************************...,2022-12-30 18:44:18.000,yingyan,"{""alarmDchatGroup"":""[{\""name\"":\""auto labeling...",vgds.dwd_autolabel_junction,104,1,dwd_autolabel_junction,dwd_autolabel_junction,101,1,1,3100,"{""commitStatus"":0_""countryGroupCode"":""""_""count...",0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1790,50093889,23495898,13,--@exclude_dependency=vgds.app_rt_trip_issue_d...,2024-02-28 19:12:41.000,haukwen,"{""alarmDchatGroup"":""""_""alarmMode"":""""_""alarmOdi...",vgds.app_rt_master_issue_stat,104,1,计算了master今日issue统计、同比昨天，环比上周,app_rt_master_issue_stat,101,1,1,3100,"{""commitStatus"":0_""countryGroupCode"":""""_""count...",0
1791,50095695,25599305,1,--@exclude_dependency=vgds.ods_all_roads_daily...,2024-02-28 20:30:25.000,halldong,"{""alarmDchatGroup"":""""_""alarmMode"":""""_""alarmOdi...",vgds.app_junction_lane_percent_di,104,1,road和lane级别的人/自驾差异指数前置数据,app_junction_lane_percent_di,101,0,1,3100,"{""commitStatus"":0_""countryGroupCode"":""""_""count...",0
1794,56645483,29648721,6,--@exclude_dependency=sparklingwater.gateway_d...,2024-07-30 11:21:34.000,jimmylimao,"{""alarmDchatGroup"":""""_""alarmMode"":""""_""alarmOdi...",vgds.app_trip_rt_event_eb_eh,104,1,Stats of EB and EH triggered times.,app_trip_rt_event_eb_eh,101,1,1,3100,"{""commitStatus"":0_""countryGroupCode"":""""_""count...",0
1795,56771577,30791975,2,--SPARK_SQL_brp_--****************************...,2024-08-01 12:31:30.000,taokexin,"{""alarmDchatGroup"":""""_""alarmMode"":""""_""alarmOdi...",vgds.dwd_ra_dotting_waypoint_and_light_assist_...,104,1,-,dwd_ra_dotting_waypoint_and_light_assist_time_...,101,1,1,3100,"{""commitStatus"":0_""countryGroupCode"":""""_""count...",0


In [27]:
sql = sql_code_df[sql_code_df['file_name'] == 'app_rt_trip_issue_detail_hf']['file_content'].iloc[0]

In [28]:
sql

"--SPARK_SQL_brp_--********************************************************************--_brp_--author:siyuanzhang_brp_--create time:2023-08-09 19:19:34_brp_--desc:trip issue 明细表 _brp_--remind:请在资源引用中添加需要引用的资源_brp_--********************************************************************--_brp_-- drop table if exists app_rt_trip_issue_detail_hf;_brp_create table if not exists `app_rt_trip_issue_detail_hf`_brp_(_brp_    `car_id` string COMMENT '车辆编号'_brp_    _`trip_comment` string COMMENT 'comment'_brp_    _`country` bigint COMMENT '1:cn 2:us'_brp_    _`driver_name` string COMMENT '驾驶员名称'_brp_    _`bag_trip_end_timestamp` bigint COMMENT 'bag_trip_end_timestamp'_brp_    _`region` string COMMENT '地区'_brp_    _`bag_trip_start_timestamp` bigint COMMENT 'bag_trip_start_timestamp'_brp_    _`trip_id` string COMMENT 'trip_id'_brp_    _`update_time` string COMMENT '更新时间'_brp_    _`user_name` string COMMENT '安全员名称'_brp_    _`test_version` string COMMENT '自动驾驶版本'_brp_    _`road_test_type` bigint COMME

In [31]:
# print(format_sql(sql))

In [42]:
from tqdm import tqdm
import os
import shutil

data_path = '/home/zhengzong/workspace/DS/HiveTrace/HiveTrace/sqllineage/data/vgds/'

## 清空data_path下的文件
shutil.rmtree(data_path)
os.makedirs(data_path)


## 遍历sql_code_df,将file_content写入data_path, 并以file_name命名
for index, row in tqdm(sql_code_df.iterrows()):
    file_name = row['file_name'] + '.sql'
    file_content = row['file_content']
    ## 格式化sql
    print(file_name)
    try:
        formatted_sql = format_sql(file_content)
    except:
        print(file_name, '格式化失败')
        print(file_content)
        continue
    
    with open(data_path + file_name, 'w') as f:
        f.write(formatted_sql)

0it [00:00, ?it/s]

1068it [00:00, 5596.56it/s]

dwd_autolabel_tp_lane_change_july.sql
get_latlon_by_pose.py.sql
route_grade_odd.sql
ego_lean_one_side_2.sql
dwd_autolabel_junction.sql
dwd_weather_district.sql
dwm_autolabel_labeled_detail.sql
app_temp_dwm_car_eff.sql
pull_status_calculate.sql
app_rt_followcar_coverage_inserthistorydata.sql
app_rt_followcar_coverage_df.sql
dm_scenario_lane_change_d.sql
dwd_asm_ego_obj_contour.sql
app_autotopic_issue_diff_detail.sql
app_ds_autotopic_accuracy.sql
dwd_release_binary.sql
dwm_voyager_car_group_on_off_line.sql
app_ds_get_latlon_by_pose.sql
holiday_for_ops.sql
dwd_autolabel_road_info.sql
dwd_sim_datasim_scenario_detail.sql
app_dwd_car_eff_temp.sql
ops_team_leader.sql
app_dwd_people_eff_temp.sql
app_dwm_car_eff_temp.sql
dim_ds_issue_pose_di.sql
dim_ds_issue_speed_di.sql
dwd_ds_mdbi_issue_new_di.sql
app_ds_mdbi_trip_issue_v4_di.sql
dwd_autolabel_rule_lane_change.sql
ce_issue_cretieria_to_hive.sql
dwd_voyager_opstrain.sql
dwd_ds_trip_exemption_st_di.sql
app_ds_mdbi_trip_issue_v3_di.sql
dwd_rtmap